# Notebook 30. IMERG precipitation and 925-hPa convergence

This is a **checkpoint-first, event-by-event** workflow for testing whether stronger low-level convergence during merged JPCZ episodes is associated with greater GPM IMERG Final V07 precipitation. It uses the current merged catalog stored in Google Drive and never changes that catalog.

## Why this layout

A prior version tried to load many ERA5 windows at once and exceeded Colab memory. This version processes a small number of events per run. After *each event*, it writes compact derived metrics to Drive. A disconnect therefore costs, at most, the event being processed—not the entire analysis.

The final analysis is built only from those compact event tables:

- **Predictor:** regional 12-hour trailing mean 925-hPa convergence at the event peak, written as $-D_{12}$ so larger values mean stronger convergence.
- **Outcomes:** IMERG `precipitationCal` event accumulation (mm) and mean precipitation rate (mm h$^{-1}$), each averaged over the JPCZ polygon and coastal wedge.
- **Test:** Pearson correlation and two-sided ordinary least-squares regression. The null hypothesis is no linear association.

IMERG Final V07 begins in June 2000, so earlier catalog events are excluded rather than assigned incomplete precipitation windows.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')
print('Repository branch:', BRANCH)
print('Drive checkpoint root:', DRIVE_ROOT)

In [ ]:
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from jpcz_catalog.config import BoundingBox, JPCZ_POLYGON_VERTICES
from jpcz_catalog.detect import compute_divergence_stack, prepare_detection_geometry
from jpcz_catalog.era5 import open_arco_era5, subset_era5_window
from jpcz_catalog.imerg import parse_imerg_granule_start, read_precipitation_cal_subset, region_mean_rates

# ----- Controls: process a small batch, inspect the printed progress, then rerun. -----
CATALOG_OVERRIDE_PATH = None  # Leave None to use the current Drive merged catalog.
RUN_ERA5_EVENT_METRICS = False
RUN_IMERG_EVENT_METRICS = False
MAX_ERA5_EVENTS_THIS_RUN = 2
MAX_IMERG_EVENTS_THIS_RUN = 1
DELETE_LOCAL_GRANULES_AFTER_CHECKPOINT = True

IMERG_FIRST_VALID_TIME = pd.Timestamp('2000-06-01 00:00:00')
MIN_TEMPORAL_COVERAGE = 0.90

COASTAL_WEDGE_VERTICES = (
    (133.05, 35.55), (136.05, 35.55), (139.55, 39.00), (139.55, 42.55),
)
REGIONS = {'jpcz_polygon': JPCZ_POLYGON_VERTICES, 'coastal_wedge': COASTAL_WEDGE_VERTICES}
ERA5_DIVERGENCE_DOMAIN = BoundingBox(lon_min=128.0, lon_max=141.0, lat_min=35.0, lat_max=43.0)
IMERG_READ_DOMAIN = BoundingBox(lon_min=128.0, lon_max=141.0, lat_min=35.0, lat_max=43.0)

DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
DRIVE_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
ERA5_EVENT_PATH = DRIVE_ANALYSIS_DIR / 'era5_925hpa_event_convergence.csv'
IMERG_RATE_PATH = DRIVE_ANALYSIS_DIR / 'imerg_regional_halfhourly_rates.csv'
IMERG_EVENT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_regional_precipitation.csv'
EVENT_METRICS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_precipitation_convergence_metrics.csv'
STATS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics.csv'
PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_scatter.png'
MANIFEST_PATH = DRIVE_ANALYSIS_DIR / 'imerg_catalog_run_manifest.csv'
LOCAL_GRANULE_DIR = Path('/content/imerg_event_granules')

def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def read_checkpoint(path, parse_dates=()):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, parse_dates=list(parse_dates))

def merge_checkpoint(existing, fresh, key):
    if existing.empty:
        merged = fresh.copy()
    elif fresh.empty:
        merged = existing.copy()
    else:
        merged = pd.concat([existing, fresh], ignore_index=True)
    return merged.drop_duplicates(key, keep='last').sort_values(key).reset_index(drop=True)

def catalog_candidates():
    return [
        Path(CATALOG_OVERRIDE_PATH) if CATALOG_OVERRIDE_PATH else None,
        DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv',
        Path('outputs/verification/jpcz_catalog_ndjf_merged_12h.csv'),
    ]

In [ ]:
catalog_path = next((path for path in catalog_candidates() if path is not None and path.exists()), None)
if catalog_path is None:
    raise FileNotFoundError('No merged catalog found. Run Notebook 06 first so the merged CSV is in Google Drive.')

catalog = pd.read_csv(catalog_path, parse_dates=['event_start', 'event_end', 'event_peak']).sort_values('event_start').reset_index(drop=True)
catalog['event_id'] = catalog['event_peak'].dt.strftime('%Y%m%dT%H%M')
if catalog['event_id'].duplicated().any():
    raise ValueError('Event peaks must be unique before the IMERG event workflow can run.')
catalog['precip_window_start'] = catalog['event_start'] - pd.Timedelta(hours=11)
catalog['precip_window_end_exclusive'] = catalog['event_end'] + pd.Timedelta(hours=1)
catalog['precip_window_hours'] = (catalog['precip_window_end_exclusive'] - catalog['precip_window_start']).dt.total_seconds() / 3600
catalog_for_imerg = catalog.loc[catalog['precip_window_start'] >= IMERG_FIRST_VALID_TIME].copy()

manifest = pd.DataFrame([{
    'catalog_source': str(catalog_path),
    'merged_catalog_rows': len(catalog),
    'imerg_eligible_events': len(catalog_for_imerg),
    'excluded_before_imerg': len(catalog) - len(catalog_for_imerg),
    'first_peak_utc': catalog['event_peak'].min(),
    'last_peak_utc': catalog['event_peak'].max(),
}])
atomic_csv(manifest, MANIFEST_PATH)

print('Catalog selected:', catalog_path)
print(f'Merged catalog: {len(catalog)} events; IMERG-eligible: {len(catalog_for_imerg)}; excluded before June 2000: {len(catalog) - len(catalog_for_imerg)}')
print('Peak-date coverage:', catalog['event_peak'].min(), 'to', catalog['event_peak'].max())
display(catalog_for_imerg[['event_id', 'event_start', 'event_end', 'event_peak', 'duration_hours', 'precip_window_hours']].head())

## 1. ERA5 convergence: one event at a time

Set `RUN_ERA5_EVENT_METRICS = True`. Each event loads only its 12-hour peak window and immediately writes one row per region to Drive. Start with the default two-event batch; rerun this cell until it reports zero remaining events.

In [ ]:
def event_convergence_row(event, arco):
    peak = pd.Timestamp(event.event_peak)
    expected = pd.date_range(peak - pd.Timedelta(hours=11), peak, freq='1h')
    window = subset_era5_window(
        arco, str(expected.min()), str(expected.max()),
        domain=ERA5_DIVERGENCE_DOMAIN,
        variables=('u_component_of_wind', 'v_component_of_wind'), level=925,
    ).sel(time=expected).load()
    geometries = {name: prepare_detection_geometry(window.longitude, window.latitude, vertices) for name, vertices in REGIONS.items()}
    divergence = compute_divergence_stack(window, dx=geometries['jpcz_polygon'].dx, dy=geometries['jpcz_polygon'].dy)
    row = {'event_id': event.event_id, 'event_peak': peak, 'convergence_window_hours': len(expected), 'status': 'ok'}
    for name, geometry in geometries.items():
        weighted = (divergence * geometry.weights).sum(dim=('latitude', 'longitude')) / geometry.weights.sum()
        d12 = float(weighted.mean('time').values)
        row[f'{name}_D12_s-1'] = d12
        row[f'{name}_convergence_1e5_s-1'] = -d12 * 1e5
    return row

era5_events = read_checkpoint(ERA5_EVENT_PATH, parse_dates=['event_peak'])
completed_ids = set(era5_events.loc[era5_events['status'].eq('ok'), 'event_id'].astype(str)) if {'event_id', 'status'}.issubset(era5_events.columns) else set()
pending_era5 = catalog_for_imerg.loc[~catalog_for_imerg['event_id'].isin(completed_ids)].copy()
print(f'ERA5 event checkpoint: {len(completed_ids)} complete; {len(pending_era5)} events remaining.')

if RUN_ERA5_EVENT_METRICS and not pending_era5.empty:
    batch = pending_era5.head(MAX_ERA5_EVENTS_THIS_RUN)
    arco = open_arco_era5(chunks={})
    for position, event in enumerate(batch.itertuples(index=False), start=1):
        print(f'ERA5 {position}/{len(batch)}: {event.event_id} | peak {event.event_peak} | loading 12 hours...')
        try:
            fresh = pd.DataFrame([event_convergence_row(event, arco)])
            era5_events = merge_checkpoint(era5_events, fresh, 'event_id')
            atomic_csv(era5_events, ERA5_EVENT_PATH)
            print(f'  saved {ERA5_EVENT_PATH.name}: {len(era5_events)} completed events')
        except Exception as error:
            print(f'  ERA5 event failed and was not marked complete: {type(error).__name__}: {error}')
        finally:
            gc.collect()
    del arco
    gc.collect()

era5_events = read_checkpoint(ERA5_EVENT_PATH, parse_dates=['event_peak'])
display(era5_events.tail())

## 2. IMERG Final: one event at a time

Set `RUN_IMERG_EVENT_METRICS = True`. On the first run, Earthdata Login opens. The notebook downloads only the half-hourly IMERG files needed for one event window, immediately reduces them to two regional rates, saves the compact time series, and then saves the event precipitation row. Start with one event to verify access and output.

In [ ]:
def precipitation_metric_row(event, rates):
    expected = pd.date_range(event.precip_window_start, event.precip_window_end_exclusive, freq='30min', inclusive='left')
    indexed = rates.set_index('time').sort_index()
    window = indexed.reindex(expected)
    row = {'event_id': event.event_id, 'event_peak': event.event_peak, 'imerg_expected_halfhours': len(expected), 'imerg_window_hours': len(expected) * 0.5}
    all_complete = True
    for name in REGIONS:
        column = f'{name}_rate_mm_hr'
        valid = int(window[column].notna().sum()) if column in window else 0
        coverage = valid / len(expected)
        accumulation = window[column].sum(skipna=True) * 0.5 if coverage >= MIN_TEMPORAL_COVERAGE and column in window else np.nan
        row[f'{name}_imerg_valid_halfhours'] = valid
        row[f'{name}_imerg_coverage_fraction'] = coverage
        row[f'{name}_imerg_accumulation_mm'] = accumulation
        row[f'{name}_imerg_mean_rate_mm_hr'] = accumulation / (len(expected) * 0.5) if pd.notna(accumulation) else np.nan
        all_complete = all_complete and pd.notna(accumulation)
    row['status'] = 'ok' if all_complete else 'incomplete'
    return row

imerg_events = read_checkpoint(IMERG_EVENT_PATH, parse_dates=['event_peak'])
completed_ids = set(imerg_events.loc[imerg_events['status'].eq('ok'), 'event_id'].astype(str)) if {'event_id', 'status'}.issubset(imerg_events.columns) else set()
pending_imerg = catalog_for_imerg.loc[~catalog_for_imerg['event_id'].isin(completed_ids)].copy()
imerg_rates = read_checkpoint(IMERG_RATE_PATH, parse_dates=['time'])
print(f'IMERG event checkpoint: {len(completed_ids)} complete; {len(pending_imerg)} events remaining. Regional-rate cache: {len(imerg_rates)} half-hours.')

if RUN_IMERG_EVENT_METRICS and not pending_imerg.empty:
    import earthaccess
    earthaccess.login()
    batch = pending_imerg.head(MAX_IMERG_EVENTS_THIS_RUN)
    for position, event in enumerate(batch.itertuples(index=False), start=1):
        expected = pd.date_range(event.precip_window_start, event.precip_window_end_exclusive, freq='30min', inclusive='left')
        existing_times = pd.DatetimeIndex(imerg_rates['time']) if not imerg_rates.empty else pd.DatetimeIndex([])
        wanted = set(expected.difference(existing_times))
        print(f'IMERG {position}/{len(batch)}: {event.event_id} | {len(wanted)}/{len(expected)} half-hours missing')
        if wanted:
            local_event_dir = LOCAL_GRANULE_DIR / event.event_id
            local_event_dir.mkdir(parents=True, exist_ok=True)
            results = earthaccess.search_data(
                short_name='GPM_3IMERGHH', version='07',
                temporal=(pd.Timestamp(event.precip_window_start).isoformat(), pd.Timestamp(event.precip_window_end_exclusive).isoformat()),
                count=500,
            )
            files = earthaccess.download(results, local_path=local_event_dir, threads=4)
            fresh_rows = []
            for file_path in files:
                try:
                    timestamp = pd.Timestamp(parse_imerg_granule_start(file_path))
                    if timestamp in wanted:
                        rate_field = read_precipitation_cal_subset(file_path, domain=IMERG_READ_DOMAIN)
                        fresh_rows.append({'time': timestamp, **{f'{name}_rate_mm_hr': value for name, value in region_mean_rates(rate_field, REGIONS).items()}})
                except Exception as error:
                    print(f'  skipped {Path(file_path).name}: {type(error).__name__}: {error}')
            if fresh_rows:
                imerg_rates = merge_checkpoint(imerg_rates, pd.DataFrame(fresh_rows), 'time')
                atomic_csv(imerg_rates, IMERG_RATE_PATH)
                print(f'  saved {len(fresh_rows)} half-hours; cache now has {len(imerg_rates)} rows')
            if DELETE_LOCAL_GRANULES_AFTER_CHECKPOINT:
                shutil.rmtree(local_event_dir, ignore_errors=True)
        metric = pd.DataFrame([precipitation_metric_row(event, imerg_rates)])
        if metric.loc[0, 'status'] == 'ok':
            imerg_events = merge_checkpoint(imerg_events, metric, 'event_id')
            atomic_csv(imerg_events, IMERG_EVENT_PATH)
            print(f'  saved complete event precipitation row: {event.event_id}')
        else:
            print('  event remains incomplete and will be retried next run.')
        gc.collect()

imerg_events = read_checkpoint(IMERG_EVENT_PATH, parse_dates=['event_peak'])
display(imerg_events.tail())

## 3. Assemble saved event rows, test the association, and plot

These cells do not call ERA5 or NASA. They only read the small Drive checkpoint tables. You can rerun them at any time as additional events are completed.

In [ ]:
era5_events = read_checkpoint(ERA5_EVENT_PATH, parse_dates=['event_peak'])
imerg_events = read_checkpoint(IMERG_EVENT_PATH, parse_dates=['event_peak'])
if not {'event_id', 'event_peak'}.issubset(era5_events.columns):
    era5_events = pd.DataFrame(columns=['event_id', 'event_peak'])
if not {'event_id', 'event_peak'}.issubset(imerg_events.columns):
    imerg_events = pd.DataFrame(columns=['event_id', 'event_peak'])
analysis = catalog_for_imerg.merge(era5_events, on=['event_id', 'event_peak'], how='left').merge(imerg_events, on=['event_id', 'event_peak'], how='left', suffixes=('_era5', '_imerg'))
atomic_csv(analysis, EVENT_METRICS_PATH)
required = [
    'jpcz_polygon_convergence_1e5_s-1', 'coastal_wedge_convergence_1e5_s-1',
    'jpcz_polygon_imerg_mean_rate_mm_hr', 'coastal_wedge_imerg_mean_rate_mm_hr',
]
missing = [column for column in required if column not in analysis.columns]
complete = 0 if missing else int(analysis.dropna(subset=required).shape[0])
print(f'Joined event table: {complete}/{len(analysis)} events have all regional convergence and IMERG rate metrics.')
if missing:
    print('Not built yet:', missing)
display(analysis.head())

In [ ]:
def association_statistics(frame, x_column, y_column, region, measure):
    if x_column not in frame or y_column not in frame:
        return {'region': region, 'precipitation_measure': measure, 'n': 0, 'status': 'checkpoint data not built yet'}
    sample = frame[[x_column, y_column]].dropna()
    n = len(sample)
    if n < 4:
        return {'region': region, 'precipitation_measure': measure, 'n': n, 'status': 'need at least four complete events'}
    result = stats.pearsonr(sample[x_column], sample[y_column])
    fit = stats.linregress(sample[x_column], sample[y_column])
    z = np.arctanh(result.statistic)
    margin = stats.norm.ppf(0.975) / np.sqrt(n - 3)
    r_low, r_high = np.tanh([z - margin, z + margin])
    slope_margin = stats.t.ppf(0.975, n - 2) * fit.stderr
    return {
        'region': region, 'precipitation_measure': measure, 'n': n, 'status': 'ok',
        'pearson_r': result.statistic, 'r_95ci_low': r_low, 'r_95ci_high': r_high, 'r_two_sided_p': result.pvalue,
        'slope': fit.slope, 'slope_95ci_low': fit.slope - slope_margin, 'slope_95ci_high': fit.slope + slope_margin,
        'slope_two_sided_p': fit.pvalue, 'intercept': fit.intercept, 'r_squared': fit.rvalue ** 2,
        'null_decision_alpha_0.05': 'reject H0' if result.pvalue < 0.05 else 'fail to reject H0',
    }

specifications = []
for region, label in [('jpcz_polygon', 'JPCZ polygon'), ('coastal_wedge', 'Coastal wedge')]:
    specifications.extend([
        (label, 'IMERG accumulation (mm)', f'{region}_convergence_1e5_s-1', f'{region}_imerg_accumulation_mm'),
        (label, 'IMERG mean rate (mm h^-1)', f'{region}_convergence_1e5_s-1', f'{region}_imerg_mean_rate_mm_hr'),
    ])
statistics_table = pd.DataFrame([association_statistics(analysis, x, y, region, measure) for region, measure, x, y in specifications])
atomic_csv(statistics_table, STATS_PATH)
display(statistics_table.round(4))

In [ ]:
def plot_association(ax, x_column, y_column, title, ylabel, summary):
    if summary.get('status') != 'ok':
        ax.text(0.5, 0.5, 'Waiting for complete checkpointed events', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return None
    sample = analysis[[x_column, y_column, 'duration_hours']].dropna()
    points = ax.scatter(sample[x_column], sample[y_column], c=sample['duration_hours'], cmap='viridis', s=38, edgecolor='white', linewidth=0.35)
    fit = stats.linregress(sample[x_column], sample[y_column])
    xline = np.linspace(sample[x_column].min(), sample[x_column].max(), 100)
    ax.plot(xline, fit.intercept + fit.slope * xline, color='#c0392b', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('925-hPa convergence, $-D_{12}$ ($10^{-5}$ s$^{-1}$)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.text(0.03, 0.97, f"n={int(summary['n'])}\nr={summary['pearson_r']:.2f} ({summary['r_95ci_low']:.2f}, {summary['r_95ci_high']:.2f})\np={summary['r_two_sided_p']:.3g}; {summary['null_decision_alpha_0.05']}", va='top', transform=ax.transAxes, fontsize=9, bbox={'facecolor': 'white', 'alpha': 0.9})
    return points

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
last_points = None
for ax, (region, measure, x_column, y_column) in zip(axes.flat, specifications):
    summary = statistics_table.loc[(statistics_table['region'] == region) & (statistics_table['precipitation_measure'] == measure)].iloc[0].to_dict()
    last_points = plot_association(ax, x_column, y_column, f'{region}: {measure}', measure, summary) or last_points
if last_points is not None:
    fig.colorbar(last_points, ax=axes, shrink=0.82, pad=0.02, label='Merged event duration (h)')
fig.suptitle('IMERG Final V07 precipitation versus ERA5 925-hPa convergence', fontsize=15)
fig.savefig(PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()
print('Figure saved:', PLOT_PATH)

## Methods wording for the presentation

For each merged JPCZ episode, we computed 925-hPa horizontal divergence from ERA5 winds and calculated the cosine-latitude-area-weighted regional mean over the digitized JPCZ polygon and coastal wedge. The predictor was the negative of the trailing 12-hour mean divergence ending at the catalog peak, so larger values represent stronger convergence. We obtained GPM IMERG Final V07 gauge-calibrated precipitation (`precipitationCal`; half-hourly 0.1° grid), calculated regional mean precipitation rates, accumulated rate × 0.5 h across each detector episode, and calculated event mean rate. We assessed the association using two-sided Pearson correlation and ordinary least-squares regression, reporting $r$, 95% confidence intervals, slope, $R^2$, and $p$ values at $\alpha=0.05$.